# 02 - Data Preprocessing: Car Plate Detection

In this notebook, we will:
1. Load and verify all Pascal VOC XML annotations
2. Split data into train / val / test sets (70 / 15 / 15)
3. Convert annotations to **YOLO format** (`.txt` files) for YOLOv8
4. Convert annotations to **COCO JSON format** for Faster R-CNN & RetinaNet
5. Preview augmentation pipeline using `albumentations`

### Output directory layout
```
data/processed/
├── yolo/
│   ├── images/
│   │   ├── train/   (303 images)
│   │   ├── val/     (65 images)
│   │   └── test/    (65 images)
│   ├── labels/
│   │   ├── train/
│   │   ├── val/
│   │   └── test/
│   └── dataset.yaml
└── coco/
    ├── images/
    │   ├── train/
    │   ├── val/
    │   └── test/
    └── annotations/
        ├── instances_train.json
        ├── instances_val.json
        └── instances_test.json
```

## 1. Setup & Imports

In [ ]:
# ── If running on Google Colab, mount Drive and install packages ──────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to wherever you uploaded the project on Drive
    PROJECT_ROOT = '/content/drive/MyDrive/license-plate-detection'
    %pip install -q albumentations opencv-python-headless
else:
    import os
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

In [ ]:
import os
import glob
import json
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

try:
    import albumentations as A
    import cv2
    ALBUMENTATIONS_OK = True
except ImportError:
    ALBUMENTATIONS_OK = False
    print('albumentations not installed — augmentation preview will be skipped.')
    print('Install with: pip install albumentations opencv-python')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')
IMAGE_DIR     = os.path.join(RAW_DIR, 'images')
ANNOT_DIR     = os.path.join(RAW_DIR, 'annotations')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

YOLO_DIR      = os.path.join(PROCESSED_DIR, 'yolo')
COCO_DIR      = os.path.join(PROCESSED_DIR, 'coco')

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15   # remainder

# Class mapping (single class for this dataset)
CLASS_NAMES = ['licence']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print('Setup complete.')
print(f'  RAW_DIR       = {RAW_DIR}')
print(f'  PROCESSED_DIR = {PROCESSED_DIR}')

## 2. Parse All Pascal VOC Annotations

In [ ]:
def parse_voc_annotation(xml_path):
    """Parse a Pascal VOC XML file.
    
    Returns a dict with image metadata and list of bounding boxes.
    Bounding box coords are absolute integers: xmin, ymin, xmax, ymax.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.find('filename').text
    size     = root.find('size')
    width    = int(size.find('width').text)
    height   = int(size.find('height').text)

    boxes = []
    for obj in root.findall('object'):
        label = obj.find('name').text.strip().lower()
        bndbox = obj.find('bndbox')
        xmin = int(float(bndbox.find('xmin').text))
        ymin = int(float(bndbox.find('ymin').text))
        xmax = int(float(bndbox.find('xmax').text))
        ymax = int(float(bndbox.find('ymax').text))

        # Clamp to image bounds
        xmin = max(0, min(xmin, width))
        ymin = max(0, min(ymin, height))
        xmax = max(0, min(xmax, width))
        ymax = max(0, min(ymax, height))

        if xmax > xmin and ymax > ymin:
            boxes.append({'label': label,
                          'xmin': xmin, 'ymin': ymin,
                          'xmax': xmax, 'ymax': ymax})

    return {'xml_path': xml_path,
            'filename': filename,
            'width': width, 'height': height,
            'boxes': boxes,
            'num_boxes': len(boxes)}


def resolve_image_path(annot, image_dir):
    """Find the actual image file on disk (handles extension mismatches)."""
    base = os.path.splitext(annot['filename'])[0]
    for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
        candidate = os.path.join(image_dir, base + ext)
        if os.path.exists(candidate):
            return candidate
    return None


# Parse all XML files
xml_paths   = sorted(glob.glob(os.path.join(ANNOT_DIR, '*.xml')))
annotations = [parse_voc_annotation(p) for p in xml_paths]

# Attach resolved image path
missing = []
for annot in annotations:
    annot['image_path'] = resolve_image_path(annot, IMAGE_DIR)
    if annot['image_path'] is None:
        missing.append(annot['filename'])

if missing:
    print(f'WARNING: {len(missing)} images not found on disk: {missing[:5]}')
    annotations = [a for a in annotations if a['image_path'] is not None]

print(f'Parsed {len(annotations)} valid annotations')
print(f'Total bounding boxes: {sum(a["num_boxes"] for a in annotations)}')
print(f'Sample: {annotations[0]}')

## 3. Train / Val / Test Split (70 / 15 / 15)

In [ ]:
def split_dataset(annotations, train_ratio=0.70, val_ratio=0.15, seed=42):
    """Stratified-ish split: keeps multi-plate images distributed evenly."""
    rng = random.Random(seed)

    # Separate single-plate and multi-plate images
    single = [a for a in annotations if a['num_boxes'] == 1]
    multi  = [a for a in annotations if a['num_boxes'] > 1]

    rng.shuffle(single)
    rng.shuffle(multi)

    def _split(lst):
        n     = len(lst)
        n_tr  = int(n * train_ratio)
        n_val = int(n * val_ratio)
        return lst[:n_tr], lst[n_tr:n_tr + n_val], lst[n_tr + n_val:]

    s_tr, s_val, s_te = _split(single)
    m_tr, m_val, m_te = _split(multi)

    train = s_tr + m_tr
    val   = s_val + m_val
    test  = s_te + m_te

    # Shuffle each split so singles and multis are interleaved
    rng.shuffle(train)
    rng.shuffle(val)
    rng.shuffle(test)

    return train, val, test


train_annots, val_annots, test_annots = split_dataset(
    annotations, TRAIN_RATIO, VAL_RATIO, SEED
)

print('Split summary')
print('=' * 40)
for name, split in [('Train', train_annots), ('Val', val_annots), ('Test', test_annots)]:
    n_boxes = sum(a['num_boxes'] for a in split)
    n_multi = sum(1 for a in split if a['num_boxes'] > 1)
    print(f'  {name:5s}: {len(split):3d} images  |  {n_boxes:3d} boxes  |  {n_multi} multi-plate images')

total = len(train_annots) + len(val_annots) + len(test_annots)
print(f'  Total: {total} images (original: {len(annotations)})')

# Visualise split proportions
fig, ax = plt.subplots(figsize=(7, 3))
sizes = [len(train_annots), len(val_annots), len(test_annots)]
labels = [f'Train\n{sizes[0]} ({sizes[0]/total*100:.0f}%)',
          f'Val\n{sizes[1]} ({sizes[1]/total*100:.0f}%)',
          f'Test\n{sizes[2]} ({sizes[2]/total*100:.0f}%)']
ax.barh(['Images'], [sizes[0]], color='steelblue', label=labels[0])
ax.barh(['Images'], [sizes[1]], left=[sizes[0]], color='coral', label=labels[1])
ax.barh(['Images'], [sizes[2]], left=[sizes[0]+sizes[1]], color='seagreen', label=labels[2])
ax.set_xlabel('Number of images')
ax.set_title('Train / Val / Test Split')
ax.legend(loc='center right')
plt.tight_layout()
plt.show()

## 4. Create Output Directory Structure

In [ ]:
SPLITS = ['train', 'val', 'test']

# YOLO dirs
for split in SPLITS:
    os.makedirs(os.path.join(YOLO_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, 'labels', split), exist_ok=True)

# COCO dirs
for split in SPLITS:
    os.makedirs(os.path.join(COCO_DIR, 'images', split), exist_ok=True)
os.makedirs(os.path.join(COCO_DIR, 'annotations'), exist_ok=True)

print('Directory structure created:')
for root, dirs, files in os.walk(PROCESSED_DIR):
    level = root.replace(PROCESSED_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

## 5. Convert to YOLO Format

YOLO expects one `.txt` file per image, where each line is:
```
<class_id>  <cx>  <cy>  <w>  <h>
```
All values are **normalised** to `[0, 1]` relative to the image dimensions.

In [ ]:
def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    """Convert VOC absolute bbox to YOLO normalised (cx, cy, w, h)."""
    cx = ((xmin + xmax) / 2) / img_w
    cy = ((ymin + ymax) / 2) / img_h
    w  = (xmax - xmin) / img_w
    h  = (ymax - ymin) / img_h
    # Clamp to [0, 1]
    cx, cy, w, h = [max(0.0, min(1.0, v)) for v in (cx, cy, w, h)]
    return cx, cy, w, h


def write_yolo_annotation(annot, label_path):
    """Write YOLO .txt label file for one image."""
    lines = []
    for box in annot['boxes']:
        class_id = CLASS_TO_ID.get(box['label'], 0)
        cx, cy, w, h = voc_to_yolo(
            box['xmin'], box['ymin'], box['xmax'], box['ymax'],
            annot['width'], annot['height']
        )
        lines.append(f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    with open(label_path, 'w') as f:
        f.write('\n'.join(lines))


def populate_yolo_split(annots, split_name, yolo_dir):
    """Copy images and write YOLO labels for one split."""
    img_out_dir = os.path.join(yolo_dir, 'images', split_name)
    lbl_out_dir = os.path.join(yolo_dir, 'labels', split_name)
    count = 0
    for annot in annots:
        src_img = annot['image_path']
        ext     = os.path.splitext(src_img)[1]
        stem    = os.path.splitext(os.path.basename(src_img))[0]

        dst_img = os.path.join(img_out_dir, stem + ext)
        dst_lbl = os.path.join(lbl_out_dir, stem + '.txt')

        shutil.copy2(src_img, dst_img)
        write_yolo_annotation(annot, dst_lbl)
        count += 1
    return count


# Run conversion for all splits
split_map = {'train': train_annots, 'val': val_annots, 'test': test_annots}
for split_name, annots in split_map.items():
    n = populate_yolo_split(annots, split_name, YOLO_DIR)
    print(f'YOLO {split_name:5s}: {n} images written')

print('\nYOLO conversion complete.')

### 5.1 Write `dataset.yaml` for YOLOv8

In [ ]:
yaml_content = f"""# YOLOv8 dataset configuration
# Generated by 02_data_preprocessing.ipynb

path: {YOLO_DIR}   # dataset root (absolute path)

train: images/train
val:   images/val
test:  images/test

nc: {len(CLASS_NAMES)}    # number of classes
names: {CLASS_NAMES}     # class names
"""

yaml_path = os.path.join(YOLO_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'Saved: {yaml_path}')
print()
print(yaml_content)

### 5.2 Verify YOLO Labels

In [ ]:
def verify_yolo_label(label_path):
    """Check that every line has 5 valid floats and values in [0,1]."""
    errors = []
    with open(label_path) as f:
        for i, line in enumerate(f, 1):
            parts = line.strip().split()
            if len(parts) != 5:
                errors.append(f'line {i}: expected 5 fields, got {len(parts)}')
                continue
            try:
                vals = list(map(float, parts))
            except ValueError:
                errors.append(f'line {i}: non-numeric value')
                continue
            if not all(0.0 <= v <= 1.0 for v in vals[1:]):
                errors.append(f'line {i}: bbox values out of [0,1] range: {vals[1:]}')
    return errors


all_ok = True
for split in SPLITS:
    lbl_dir = os.path.join(YOLO_DIR, 'labels', split)
    label_files = glob.glob(os.path.join(lbl_dir, '*.txt'))
    bad = []
    for lf in label_files:
        errs = verify_yolo_label(lf)
        if errs:
            bad.append((os.path.basename(lf), errs))
    status = 'OK' if not bad else f'{len(bad)} files with errors'
    print(f'  {split:5s} labels ({len(label_files)} files): {status}')
    if bad:
        all_ok = False
        for fname, errs in bad[:3]:
            print(f'    {fname}: {errs}')

print()
print('All YOLO labels valid!' if all_ok else 'Some labels have issues — review above.')

### 5.3 Visualise a YOLO-format Sample

In [ ]:
def yolo_to_abs(cx, cy, w, h, img_w, img_h):
    """Convert normalised YOLO bbox back to absolute pixel coords."""
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return int(x1), int(y1), int(x2), int(y2)


# Pick 6 random images from the training split
sample_annots = random.sample(train_annots, min(6, len(train_annots)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, annot in zip(axes.flatten(), sample_annots):
    stem = os.path.splitext(os.path.basename(annot['image_path']))[0]
    ext  = os.path.splitext(annot['image_path'])[1]
    img_path = os.path.join(YOLO_DIR, 'images', 'train', stem + ext)
    lbl_path = os.path.join(YOLO_DIR, 'labels', 'train', stem + '.txt')

    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]

    ax.imshow(img)
    with open(lbl_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            cls_id, cx, cy, bw, bh = int(parts[0]), parts[1], parts[2], parts[3], parts[4]
            x1, y1, x2, y2 = yolo_to_abs(cx, cy, bw, bh, w, h)
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, CLASS_NAMES[cls_id], color='lime',
                    fontsize=9, fontweight='bold', backgroundcolor='black')
    ax.set_title(stem, fontsize=9)
    ax.axis('off')

plt.suptitle('YOLO Format Verification — Training Split', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Convert to COCO JSON Format

COCO JSON uses the `[x, y, width, height]` convention for bounding boxes  
(top-left corner, absolute pixel values).

In [ ]:
def build_coco_json(annots, split_name, coco_dir, class_names):
    """
    Build a COCO-format JSON dict and copy images into the split folder.

    COCO bbox format: [x_min, y_min, width, height]  (absolute pixels)
    """
    categories = [
        {'id': idx + 1, 'name': name, 'supercategory': 'vehicle'}
        for idx, name in enumerate(class_names)
    ]

    coco_images      = []
    coco_annotations = []
    annotation_id    = 1

    img_out_dir = os.path.join(coco_dir, 'images', split_name)

    for image_id, annot in enumerate(annots, start=1):
        src_img = annot['image_path']
        ext     = os.path.splitext(src_img)[1]
        stem    = os.path.splitext(os.path.basename(src_img))[0]
        dst_img = os.path.join(img_out_dir, stem + ext)
        shutil.copy2(src_img, dst_img)

        coco_images.append({
            'id':        image_id,
            'file_name': stem + ext,
            'width':     annot['width'],
            'height':    annot['height'],
        })

        for box in annot['boxes']:
            bw = box['xmax'] - box['xmin']
            bh = box['ymax'] - box['ymin']
            cat_id = class_names.index(box['label']) + 1  # COCO ids are 1-based

            coco_annotations.append({
                'id':           annotation_id,
                'image_id':     image_id,
                'category_id':  cat_id,
                'bbox':         [box['xmin'], box['ymin'], bw, bh],
                'area':         bw * bh,
                'iscrowd':      0,
                'segmentation': [],
            })
            annotation_id += 1

    coco_dict = {
        'info':        {'description': 'License Plate Detection',
                        'version': '1.0', 'year': 2024},
        'licenses':    [],
        'categories':  categories,
        'images':      coco_images,
        'annotations': coco_annotations,
    }

    out_path = os.path.join(coco_dir, 'annotations', f'instances_{split_name}.json')
    with open(out_path, 'w') as f:
        json.dump(coco_dict, f, indent=2)

    return coco_dict, out_path


coco_results = {}
for split_name, annots in split_map.items():
    coco_dict, out_path = build_coco_json(annots, split_name, COCO_DIR, CLASS_NAMES)
    coco_results[split_name] = coco_dict
    print(f'COCO {split_name:5s}: {len(coco_dict["images"]):3d} images, '
          f'{len(coco_dict["annotations"]):3d} annotations  →  {out_path}')

print('\nCOCO JSON conversion complete.')

### 6.1 Verify COCO JSON Structure

In [ ]:
def verify_coco_json(coco_dict):
    """Basic sanity checks on a COCO dict."""
    errors = []

    # Every annotation must reference a valid image id
    image_ids = {img['id'] for img in coco_dict['images']}
    for ann in coco_dict['annotations']:
        if ann['image_id'] not in image_ids:
            errors.append(f'annotation {ann["id"]} references unknown image_id {ann["image_id"]}')

    # bbox must be non-degenerate
    for ann in coco_dict['annotations']:
        x, y, w, h = ann['bbox']
        if w <= 0 or h <= 0:
            errors.append(f'annotation {ann["id"]} has zero-area bbox: {ann["bbox"]}')

    return errors


print('COCO JSON verification')
all_ok = True
for split, coco_dict in coco_results.items():
    errs = verify_coco_json(coco_dict)
    status = 'OK' if not errs else f'{len(errs)} errors'
    print(f'  {split:5s}: {status}')
    if errs:
        all_ok = False
        for e in errs[:5]:
            print(f'    {e}')

# Print structure of one sample annotation
sample_ann = coco_results['train']['annotations'][0]
print(f'\nSample COCO annotation:')
print(json.dumps(sample_ann, indent=2))

print()
print('All COCO JSONs valid!' if all_ok else 'Some COCO JSONs have issues — see above.')

### 6.2 Visualise a COCO-format Sample

In [ ]:
coco_train = coco_results['train']
id_to_image = {img['id']: img for img in coco_train['images']}

# Group annotations by image_id
anns_by_img = defaultdict(list)
for ann in coco_train['annotations']:
    anns_by_img[ann['image_id']].append(ann)

sample_ids = random.sample(list(id_to_image.keys()), min(6, len(id_to_image)))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, img_id in zip(axes.flatten(), sample_ids):
    img_info = id_to_image[img_id]
    img_path = os.path.join(COCO_DIR, 'images', 'train', img_info['file_name'])
    img = np.array(Image.open(img_path))
    ax.imshow(img)

    for ann in anns_by_img[img_id]:
        x, y, w, h = ann['bbox']
        rect = patches.Rectangle((x, y), w, h,
                                  linewidth=2, edgecolor='cyan', facecolor='none')
        ax.add_patch(rect)
        cat_name = next(c['name'] for c in coco_train['categories']
                        if c['id'] == ann['category_id'])
        ax.text(x, y - 5, cat_name, color='cyan',
                fontsize=9, fontweight='bold', backgroundcolor='black')

    ax.set_title(img_info['file_name'], fontsize=9)
    ax.axis('off')

plt.suptitle('COCO Format Verification — Training Split', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Data Augmentation Pipeline Preview

We define the augmentation pipeline here for **documentation purposes**.  
YOLOv8 applies its own augmentations during training.  
For Faster R-CNN / RetinaNet we'll apply this pipeline manually in the training notebook.

> **Note:** Augmentations are applied *at training time* (online), **not** baked into the split files.

In [ ]:
if not ALBUMENTATIONS_OK:
    print('Skipping augmentation preview — albumentations not installed.')
else:
    # ── Augmentation pipeline ──────────────────────────────────────────────────
    # bbox_params: albumentations uses 'pascal_voc' (xmin,ymin,xmax,ymax) format
    #
    # Note: RandomResizedCrop is omitted — its API changes across albumentations
    # versions. ShiftScaleRotate(scale_limit=0.15) covers the same effect.
    train_transform = A.Compose(
        [
            # Geometric
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                               rotate_limit=10, p=0.5,
                               border_mode=cv2.BORDER_CONSTANT),
            # Photometric
            A.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.3, hue=0.1, p=0.5),
            A.GaussNoise(std_range=(0.03, 0.15), p=0.3),
            A.MotionBlur(blur_limit=5, p=0.2),
            A.CLAHE(clip_limit=4.0, p=0.2),
            # Dropout / occlusion
            A.CoarseDropout(num_holes_range=(1, 8),
                            hole_height_range=(8, 32),
                            hole_width_range=(8, 32),
                            fill_value=0, p=0.2),
        ],
        bbox_params=A.BboxParams(
            format='pascal_voc',
            label_fields=['category_ids'],
            min_visibility=0.3,
        )
    )

    val_transform = A.Compose(
        [A.Resize(height=640, width=640)],
        bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category_ids'])
    )

    print('Augmentation pipelines defined.')
    print()
    print('Train transforms:')
    for t in train_transform.transforms:
        print(f'  - {t.__class__.__name__}')
    print()
    print('Val transform:')
    for t in val_transform.transforms:
        print(f'  - {t.__class__.__name__}')

### 7.1 Visualise Augmented Samples

In [ ]:
if not ALBUMENTATIONS_OK:
    print('Skipping — albumentations not installed.')
else:
    # Pick one training image and show 8 augmented versions
    ref_annot = random.choice(train_annots)
    src_img   = cv2.cvtColor(cv2.imread(ref_annot['image_path']), cv2.COLOR_BGR2RGB)
    src_boxes = [[b['xmin'], b['ymin'], b['xmax'], b['ymax']] for b in ref_annot['boxes']]
    src_cats  = [0] * len(src_boxes)

    fig, axes = plt.subplots(2, 4, figsize=(20, 9))

    # Original
    ax = axes[0, 0]
    ax.imshow(src_img)
    for x1, y1, x2, y2 in src_boxes:
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    ax.set_title('Original', fontweight='bold')
    ax.axis('off')

    # 7 augmented versions
    for ax in axes.flatten()[1:]:
        result = train_transform(
            image=src_img.copy(),
            bboxes=src_boxes,
            category_ids=src_cats
        )
        aug_img   = result['image']
        aug_boxes = result['bboxes']

        ax.imshow(aug_img)
        for x1, y1, x2, y2 in aug_boxes:
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
        ax.set_title('Augmented')
        ax.axis('off')

    plt.suptitle(f'Augmentation Preview — {os.path.basename(ref_annot["image_path"])}',
                 fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Final Summary & Verification

In [ ]:
print('=' * 60)
print('PREPROCESSING SUMMARY')
print('=' * 60)

print(f'\nDataset split (seed={SEED}):')
for split_name, annots in split_map.items():
    boxes_count = sum(a['num_boxes'] for a in annots)
    print(f'  {split_name:5s}: {len(annots):3d} images  |  {boxes_count:3d} bounding boxes')

print(f'\nYOLO format  →  {YOLO_DIR}')
for split in SPLITS:
    img_dir = os.path.join(YOLO_DIR, 'images', split)
    lbl_dir = os.path.join(YOLO_DIR, 'labels', split)
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f'  {split:5s}: {n_img} images, {n_lbl} label files')

print(f'\nCOCO format  →  {COCO_DIR}')
for split in SPLITS:
    json_path = os.path.join(COCO_DIR, 'annotations', f'instances_{split}.json')
    size_kb   = os.path.getsize(json_path) // 1024
    with open(json_path) as f:
        d = json.load(f)
    print(f'  {split:5s}: {len(d["images"]):3d} images, '
          f'{len(d["annotations"]):3d} annotations  ({size_kb} KB)')

print(f'\nYOLO dataset.yaml: {os.path.join(YOLO_DIR, "dataset.yaml")}')

print()
print('=' * 60)
print('NEXT: 03_train_yolov8.ipynb')
print('      Run on Google Colab with GPU runtime')
print('=' * 60)

---
## Next Steps

All data is now ready.  In the next notebooks:

| Notebook | Content |
|---|---|
| `03_train_yolov8.ipynb` | Train YOLOv8n/s on the YOLO-format data using `ultralytics` |
| `04_train_faster_rcnn.ipynb` | Train Faster R-CNN using `torchvision` and the COCO JSON files |
| `05_train_retinanet.ipynb` | Train RetinaNet using `torchvision` and the COCO JSON files |
| `06_evaluation.ipynb` | Compare all models on mAP\@50, mAP\@50:95, speed (FPS) |

### Colab tip
Upload the entire `data/processed/` folder to Google Drive, mount it in Colab,  
and update `YOLO_DIR` / `COCO_DIR` paths accordingly.